# Decile Analysis: Gains, Lift, KS and Rank Ordering

Wiki reference for [Decile analysis, gains and lift](https://ml-viz-ruby.vercel.app/wiki/decile-analysis-and-lift).

**The idea in one sentence.** Sort the population by score, cut it into equal-count bins, and count the outcomes in each: the resulting table says where the risk actually landed, which a single AUC never does.

**Why it matters.** "Action the worst 10%" is an operational decision with a budget attached. The decile table turns a ranking metric into capture rates, lift and a cutoff, and it is the first exhibit in a scorecard validation pack, a churn review or a campaign sizing deck.

*Colab tip: File → Save a copy in Drive before editing.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score

plt.style.use('dark_background')
rng = np.random.default_rng(7)
pd.set_option('display.width', 120)

## 1 — From scratch: the table

A synthetic book of 10,000 accounts with a 5% bad rate. Bads and goods are drawn from two overlapping normals, so the model discriminates but far from perfectly.

In [ ]:
N, BASE_RATE = 10_000, 0.05
n_bad = int(N * BASE_RATE)
n_good = N - n_bad

score = np.concatenate([rng.normal(1.1, 1.0, n_bad), rng.normal(0.0, 1.0, n_good)])
bad   = np.concatenate([np.ones(n_bad, int),         np.zeros(n_good, int)])

print(f'{N:,} accounts, {n_bad} bad ({bad.mean():.1%} base rate)')
print(f'AUC = {roc_auc_score(bad, score):.4f}')

The construction is four steps: sort descending, cut into equal-*count* bins, count per bin, accumulate from the top.

$$\text{lift}_i = \frac{b_i/n_i}{B_k/N}, \qquad \text{gains}_i = \frac{B_i}{B_k}, \qquad \mathrm{KS}_i = \frac{B_i}{B_k} - \frac{G_i}{G_k}$$

In [ ]:
def decile_table(score, bad, k=10):
    """Equal-count bins, riskiest first, with lift / gains / KS columns."""
    order = np.argsort(-score)                   # descending: riskiest first
    b = bad[order]
    bins = np.array_split(np.arange(len(b)), k)  # equal-count, not equal-width

    rows = []
    for i, idx in enumerate(bins, start=1):
        rows.append({'bin': i, 'n': len(idx), 'bads': int(b[idx].sum()),
                     'min_score': score[order][idx].min()})
    t = pd.DataFrame(rows)

    t['goods']     = t['n'] - t['bads']
    t['bad_rate']  = t['bads'] / t['n']
    t['lift']      = t['bad_rate'] / bad.mean()
    t['cum_bad']   = t['bads'].cumsum() / t['bads'].sum()
    t['cum_good']  = t['goods'].cumsum() / t['goods'].sum()
    t['cum_lift']  = (t['bads'].cumsum() / t['n'].cumsum()) / bad.mean()
    t['ks']        = t['cum_bad'] - t['cum_good']
    return t

table = decile_table(score, bad, k=10)
print(table[['bin', 'n', 'bads', 'bad_rate', 'lift',
             'cum_bad', 'cum_good', 'cum_lift', 'ks']].round(4).to_string(index=False))

In [ ]:
ks_row = table.loc[table['ks'].idxmax()]
print(f"top-decile lift    : {table.loc[0, 'lift']:.2f}x")
print(f"capture at 20%     : {table.loc[1, 'cum_bad']:.1%} of all bads")
print(f"KS                 : {ks_row['ks']:.4f} at bin {int(ks_row['bin'])}")
print(f"monotone bad rate? : {bool((table['bad_rate'].diff().dropna() <= 0).all())}")

## 2 — The library way, cross-checked

There is no `sklearn.decile_table`, but every column has a library counterpart to check against. The KS column is the two-sample Kolmogorov–Smirnov statistic evaluated on the decile grid, and the cumulative curves are the ROC curve sampled at ten points.

In [ ]:
# KS: ours (on the decile grid) vs scipy's exact two-sample statistic.
exact_ks = stats.ks_2samp(score[bad == 1], score[bad == 0]).statistic
print(f'binned KS (10 bins): {table["ks"].max():.4f}')
print(f'exact KS (scipy)   : {exact_ks:.4f}')
assert table['ks'].max() <= exact_ks + 1e-12, 'a bin grid cannot beat the exact sup'

# Gini from the decile grid vs the exact pairwise AUC.
x = np.r_[0, table['cum_good'].to_numpy()]
y = np.r_[0, table['cum_bad'].to_numpy()]
auc_grid = np.trapezoid(y, x)
auc_exact = roc_auc_score(bad, score)
print(f'\nGini from 10 points: {2 * auc_grid - 1:.4f}  (AUC {auc_grid:.4f})')
print(f'Gini exact         : {2 * auc_exact - 1:.4f}  (AUC {auc_exact:.4f})')

The binned KS is below the exact one, as it must be: a ten-point grid can only find the gap at its own cut points. The decile Gini is close to the exact one but not equal, because the trapezoid interpolates straight lines across bins. Report the exact figures and treat the table's versions as arithmetic cross-checks.

## 3 — Visualize it: gains, lift, KS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
pop = np.r_[0, table['bin'].to_numpy() / len(table)]
ax.plot(pop, np.r_[0, table['cum_bad']], 'o-', color='#14b8a6', label='bads captured')
ax.plot(pop, np.r_[0, table['cum_good']], 'o-', color='#6366f1', label='goods captured')
ax.plot([0, 1], [0, 1], '--', color='#475569', label='random targeting')
kb = int(ks_row['bin'])
ax.vlines(kb / len(table), ks_row['cum_good'], ks_row['cum_bad'],
          color='#eab308', lw=2.5)
ax.annotate(f"KS = {ks_row['ks']:.3f}", (kb / len(table) + 0.02,
            (ks_row['cum_bad'] + ks_row['cum_good']) / 2), color='#eab308')
ax.set_xlabel('% of population targeted'); ax.set_ylabel('cumulative % captured')
ax.set_title('Cumulative gains'); ax.legend(loc='lower right')

ax = axes[1]
colors = ['#14b8a6' if l >= 1 else '#6366f1' for l in table['lift']]
ax.bar(table['bin'], table['lift'], color=colors)
ax.axhline(1.0, ls='--', color='#94a3b8')
ax.set_xlabel('decile (1 = riskiest)'); ax.set_ylabel('lift')
ax.set_title('Lift per decile')
plt.tight_layout(); plt.show()

**What to notice.** The teal gains curve bows hard toward the top-left: the first two deciles hold most of the bads. The widest vertical gap between the two cumulative curves is the KS, and the decile where it occurs is the natural cutoff candidate. On the right, every bar above the dashed line is a bin that beats random targeting, and the bars must descend if the score is monotone in risk.

### What prevalence does to lift

Now the part that catches people out. Hold the **model** fixed and change only the base rate. AUC and KS are rank statistics and barely move; lift is a ratio *to the base rate* and collapses, because a bin holding 10% of the population cannot lift more than $\min(1/\pi,\ k)$ for $k$ bins: at a 25% base rate that ceiling is 4, and the decile itself caps it at 10 however rare the bads are.

In [ ]:
def resample_at_prevalence(pi, n_total=10_000, seed=11):
    r = np.random.default_rng(seed)
    nb = int(n_total * pi)
    s = np.concatenate([r.normal(1.1, 1.0, nb), r.normal(0.0, 1.0, n_total - nb)])
    y = np.concatenate([np.ones(nb, int), np.zeros(n_total - nb, int)])
    return s, y

print(f"{'prevalence':>11} {'AUC':>7} {'KS':>7} {'top lift':>9} {'ceiling':>9}")
for pi in (0.02, 0.05, 0.10, 0.25, 0.50):
    s, y = resample_at_prevalence(pi)
    t = decile_table(s, y, k=10)
    print(f'{pi:>10.0%} {roc_auc_score(y, s):>7.3f} {t["ks"].max():>7.3f} '
          f'{t.loc[0, "lift"]:>9.2f} {min(1 / pi, 10):>9.1f}')

The AUC column moves by 0.01 across a 25× change in base rate, while top-decile lift falls by more than half. A lift figure quoted without its prevalence is not a claim about a model.

## 4 — Tradeoffs and when to use it

| Metric | Answers | Depends on prevalence? | Watch out for |
|---|---|---|---|
| **AUC / Gini** | how well does it rank? | no | blind to calibration and to where the cutoff is |
| **KS** | how separated are the two score distributions? | barely | binned KS ≤ exact KS; the p-value is meaningless here |
| **Lift** | how much better than random targeting? | **yes**, capped at $1/\pi$ | never compare across base rates |
| **Gains / capture** | what do I catch at this budget? | yes, weakly | the whole point of the table |
| **Rank-order breaks** | is the score monotone in risk? | no | few bads per bin makes breaks look real |

**Failure modes**

1. **Equal-width instead of equal-count bins**: a skewed score leaves the top "decile" with 40 accounts and pure noise in every rate.
2. **Re-cutting deciles each monitoring period**: a population shift then reads as a risk change. Freeze the development cut points and apply those score thresholds later.
3. **Too few bads at the bottom**: a "break" between two bins holding 5 and 8 bads is well inside sampling noise.
4. **Quoting the decile Gini as the model's AUC**: it is a ten-point approximation.

In [ ]:
# Freeze development cut points, then apply them to a drifted population.
dev_cuts = np.quantile(score, np.linspace(0, 1, 11))[1:-1]   # 9 interior edges

drift = score + 0.35                      # everyone scores higher next month
print('accounts per bin, development cuts applied to both periods')
print('dev   :', np.bincount(np.digitize(score, dev_cuts), minlength=10))
print('drifted:', np.bincount(np.digitize(drift, dev_cuts), minlength=10))
print('\nRe-cutting on the drifted population would hide this entirely:')
print('re-cut :', np.bincount(np.digitize(
    drift, np.quantile(drift, np.linspace(0, 1, 11))[1:-1]), minlength=10))

## 5 — ✏️ Your turn

### Exercise 1: read the table

Given per-decile bad counts on a 10,000-account book with 500 bads, compute the top-decile lift and the capture rate of the top three deciles, without building a DataFrame.

In [ ]:
bads_per_decile = np.array([185, 110, 70, 45, 30, 22, 15, 10, 8, 5])
n_per_decile = 1000

def exercise_1(bads, n_bin, n_total):
    base_rate = None      # TODO(you): overall bad rate
    top_lift = None       # TODO(you): decile 1 bad rate / base rate
    capture_top3 = None   # TODO(you): share of ALL bads in the first three deciles
    return base_rate, top_lift, capture_top3

In [ ]:
base, lift1, cap3 = exercise_1(bads_per_decile, n_per_decile, 10_000)
assert abs(base - 0.05) < 1e-12, f'base rate should be 0.05, got {base}'
assert abs(lift1 - 3.70) < 1e-9, f'top-decile lift should be 3.70, got {lift1}'
assert abs(cap3 - 0.73) < 1e-9, f'top-3 capture should be 0.73, got {cap3}'
print(f'base rate {base:.1%}, top-decile lift {lift1:.2f}x, top-3 capture {cap3:.0%}')

<details>
<summary>Solution</summary>

```python
def exercise_1(bads, n_bin, n_total):
    base_rate = bads.sum() / n_total            # 500 / 10000 = 0.05
    top_lift = (bads[0] / n_bin) / base_rate    # 0.185 / 0.05 = 3.70
    capture_top3 = bads[:3].sum() / bads.sum()  # 365 / 500 = 0.73
    return base_rate, top_lift, capture_top3
```

Reading it out: the riskiest tenth of the book defaults at 3.7× the book average, and reviewing the top three deciles reaches 73% of everything that will go bad.
</details>

### Exercise 2: Hosmer–Lemeshow on the same bins

The decile table measures discrimination. Reuse its bins to test **calibration**: compare observed bads per bin against what the model's own predicted probabilities expected.

$$\hat{C} = \sum_{i=1}^{g} \frac{(O_i - E_i)^2}{E_i(1 - \bar{p}_i)}, \qquad E_i = n_i \bar{p}_i$$

with $g - 2$ degrees of freedom when the bins come from a model fit on the same data.

In [ ]:
observed  = np.array([185, 110, 70, 45, 30, 22, 15, 10, 8, 5])
mean_pred = np.array([0.170, 0.115, 0.074, 0.048, 0.031,
                      0.021, 0.014, 0.010, 0.007, 0.004])
n_bin = 1000

def hosmer_lemeshow(observed, mean_pred, n_bin):
    expected = None     # TODO(you): n_bin * mean_pred, per bin
    stat = None         # TODO(you): sum of (O - E)^2 / (E * (1 - p_bar))
    dof = None          # TODO(you): g - 2
    p_value = None      # TODO(you): stats.chi2.sf(stat, dof)
    return stat, dof, p_value

In [ ]:
stat, dof, p = hosmer_lemeshow(observed, mean_pred, n_bin)
assert abs(stat - 2.8199437893) < 1e-6, f'expected 2.8199, got {stat}'
assert dof == 8, f'expected 8 degrees of freedom, got {dof}'
assert abs(p - 0.9451437452) < 1e-6, f'expected p = 0.9451, got {p}'

contrib = (observed - n_bin * mean_pred) ** 2 / (n_bin * mean_pred * (1 - mean_pred))
print(f'HL statistic {stat:.4f} on {dof} df, p = {p:.4f}')
print(f'decile 1 alone contributes {contrib[0] / stat:.0%} of it')

<details>
<summary>Solution</summary>

```python
def hosmer_lemeshow(observed, mean_pred, n_bin):
    expected = n_bin * mean_pred
    stat = np.sum((observed - expected) ** 2 / (expected * (1 - mean_pred)))
    dof = len(observed) - 2
    p_value = stats.chi2.sf(stat, dof)
    return stat, dof, p_value
```

$\hat{C} = 2.82$, $p = 0.945$: no evidence of miscalibration. Decile 1 supplies more than half the statistic, which is the genuinely useful output. Hosmer–Lemeshow is nearly powerless on small samples and rejects almost everything on large ones, so read the per-bin residuals rather than the verdict.
</details>

## 6 — Key takeaways

- **Equal-count bins, sorted by score.** Bad rate, lift, gains and KS all come from the same four counts per bin.
- **Lift is normalised by prevalence** and capped at $1/\pi$; AUC and KS are not. Never compare lift across populations with different base rates.
- **Binned KS ≤ exact KS**, and the decile Gini is a ten-point approximation to the pairwise AUC. Say which you computed.
- **Rank-order breaks are findings**, unless the bin holds too few bads to estimate a rate.
- **Freeze development cut points** for monitoring, otherwise drift becomes invisible.

**Next:** [concordance and rank statistics](https://ml-viz-ruby.vercel.app/wiki/concordance-and-rank-statistics) for the exact c-statistic the table approximates, and [comparing model performance](https://ml-viz-ruby.vercel.app/wiki/comparing-model-performance) for testing one table against another.